# Orbital PPO basic

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import numba as nb
import numpy as np
import xarray.ufuncs as xrf
import matplotlib.pyplot as plt
import pickle
from itertools import product

from tqdm.auto import trange, tqdm
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets

from ray import tune
from ray.tune.registry import register_env
from ray.rllib.agents import ppo
from ray.rllib.models import ModelCatalog
from ray.rllib.models.tf.tf_modelv2 import TFModelV2
from ray.rllib.models.tf.fcnet import FullyConnectedNetwork
from ray.tune.utils.log import Verbosity
from ray.tune import JupyterNotebookReporter
# Verbosity.V0_MINIMAL
# Verbosity.V1_EXPERIMENT
# Verbosity.V2_TRIAL_NORM
# Verbosity.V3_TRIAL_DETAILS

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import LauncherV1Orbital, Stage, AP_PITCH_RATE_CONTROL, wrap_angle
from sufcnet import SUFullyConnectedNetwork

## Configure

In [3]:
# tensorboard --logdir=~/ray_results --port=8270 --host=0.0.0.0

In [4]:
register_env("LauncherV1Orbital", LauncherV1Orbital)
ModelCatalog.register_custom_model("SUFullyConnectedNetwork", SUFullyConnectedNetwork)

## Learn

In [5]:
tune.run(
    ppo.PPOTrainer,
    config={
        "gamma": 0.995,
        "vf_clip_param": 1.,
        "clip_param": 0.2,
        
        "env": "LauncherV1Orbital",
        "num_workers": 4,
        "num_gpus": 0,
        "batch_mode": "complete_episodes",
        "env_config": {},
        "model": {
            "custom_model": "SUFullyConnectedNetwork",
            "custom_model_config": {
                "dropouts": [0.1, 0.1, 0.1],
                "training": True},
            "fcnet_hiddens": [64, 64, 64],
            "fcnet_activation": "relu",
        },
    },
    checkpoint_freq=1,
    checkpoint_at_end=True,
    local_dir="./results",
    name="ppo_basic",
    progress_reporter=JupyterNotebookReporter(True),
    verbose=3,
#     resume=True,
#     restore="~/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_8e488_00000_0_2021-05-15_01-19-12/checkpoint_001230",
)

TuneError: ('Trials did not complete', [PPOTrainer_LauncherV1Orbital_26bba_00000])